# Feature Drop Scoping Check

Reads the manual feature-decision spreadsheet and drops every column marked exactly `"Drop"`, for both datasets. Loads Stage 1's output (tmdb_stage1.parquet/airbnb_stage1.parquet), not the raw loaders -- this is a scoping check only, no output file is saved here.

## Setup

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

INTERIM_DIR = REPO_ROOT / "data" / "interim"
FEATURES_XLSX_PATH = REPO_ROOT.parent / "Feature Decisions.xlsx"
FIGURES_DIR = REPO_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def savefig(fig, filename):
    path = FIGURES_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight")
    print(f"Saved: {path}")

## Load Stage 1 datasets

In [ ]:
tmdb = pd.read_parquet(INTERIM_DIR / "tmdb_stage1.parquet")
airbnb = pd.read_parquet(INTERIM_DIR / "airbnb_stage1.parquet")
print(f"Loaded tmdb_stage1.parquet: {tmdb.shape}")
print(f"Loaded airbnb_stage1.parquet: {airbnb.shape}")

## Load feature decisions

In [ ]:
tmdb_decisions = pd.read_excel(FEATURES_XLSX_PATH, sheet_name="TMDB")
airbnb_decisions = pd.read_excel(FEATURES_XLSX_PATH, sheet_name="Airbnb")
print("TMDB decision values:", tmdb_decisions["Decision"].value_counts().to_dict())
print("Airbnb decision values:", airbnb_decisions["Decision"].value_counts().to_dict())

## Drop columns marked exactly "Drop"

`id` (both datasets) and `imdb_id` (TMDB only -- Airbnb has no such column) are never dropped even where marked "Drop" -- id is needed to keep track of records, and imdb_id is kept per an explicit instruction not to remove it. Any decision other than exactly "Drop" (e.g. "Drop - else target", "Drop - change to binary", "Drop - condition") is left untouched here; those need separate handling, not a plain column drop.

In [ ]:
def cols_to_drop(decisions_df, always_keep):
    drop_mask = decisions_df["Decision"] == "Drop"
    features = decisions_df.loc[drop_mask, "Feature"].tolist()
    return [f for f in features if f not in always_keep]


tmdb_drop_cols = cols_to_drop(tmdb_decisions, always_keep=["id", "imdb_id"])
airbnb_drop_cols = cols_to_drop(airbnb_decisions, always_keep=["id"])

print(f"TMDB: {len(tmdb_drop_cols)} columns to drop")
print(tmdb_drop_cols)
print()
print(f"Airbnb: {len(airbnb_drop_cols)} columns to drop")
print(airbnb_drop_cols)

In [ ]:
tmdb_shape_before = tmdb.shape
tmdb_dropped = tmdb.drop(columns=tmdb_drop_cols)
tmdb_shape_after = tmdb_dropped.shape
print(f"TMDB: {tmdb_shape_before} -> {tmdb_shape_after}")

airbnb_shape_before = airbnb.shape
airbnb_dropped = airbnb.drop(columns=airbnb_drop_cols)
airbnb_shape_after = airbnb_dropped.shape
print(f"Airbnb: {airbnb_shape_before} -> {airbnb_shape_after}")

## TMDB feature engineering

### Drop raw list-string columns superseded by their `_list` versions

In [ ]:
raw_list_cols = ["genres", "keywords", "production_companies", "production_countries", "spoken_languages"]
tmdb_list_cols_to_drop = [c for c in raw_list_cols if f"{c}_list" in tmdb_dropped.columns and c in tmdb_dropped.columns]
tmdb_dropped = tmdb_dropped.drop(columns=tmdb_list_cols_to_drop)
print(f"Dropped (superseded by *_list): {tmdb_list_cols_to_drop}")
print(f"TMDB shape now: {tmdb_dropped.shape}")

### Feature names & dtypes

In [ ]:
tmdb_dropped.dtypes

### Remove inconsistent vote_average/vote_count rows

`vote_average`/`vote_count` is marked `"Drop - else target"` in the decision sheet, deferred from the plain-drop step. This resolves the logically-impossible combinations found earlier: a nonzero average with zero votes, or a zero average despite having real votes -- both are corrupted, not genuine data points.

In [ ]:
tmdb_shape_before_vote_fix = tmdb_dropped.shape
bad_vote_mask = (
    ((tmdb_dropped["vote_average"] != 0) & (tmdb_dropped["vote_count"] == 0))
    | ((tmdb_dropped["vote_average"] == 0) & (tmdb_dropped["vote_count"] != 0))
)
tmdb_dropped = tmdb_dropped[~bad_vote_mask]
print(f"Dropped {int(bad_vote_mask.sum())} rows with inconsistent vote_average/vote_count")
print(f"Shape: {tmdb_shape_before_vote_fix} -> {tmdb_dropped.shape}")

### Drop records with missing title

In [ ]:
tmdb_shape_before_title_drop = tmdb_dropped.shape
tmdb_dropped = tmdb_dropped[tmdb_dropped["title"].notna()]
print(f"Dropped {tmdb_shape_before_title_drop[0] - tmdb_dropped.shape[0]} rows with title == NaN")
print(f"Shape: {tmdb_shape_before_title_drop} -> {tmdb_dropped.shape}")

### Missing values

In [ ]:
tmdb_missing = pd.DataFrame({
    "n_missing": tmdb_dropped.isnull().sum(),
    "pct_missing": tmdb_dropped.isnull().mean() * 100,
}).sort_values("pct_missing", ascending=False)
tmdb_missing

### Filter to adult == False

In [ ]:
tmdb_shape_before_adult_filter = tmdb_dropped.shape
tmdb_dropped = tmdb_dropped[tmdb_dropped["adult"] == False]
print(f"Shape: {tmdb_shape_before_adult_filter} -> {tmdb_dropped.shape}")

## TMDB candidate DataFrames (post adult filter, pre downstream drops)

Branches off from `tmdb_dropped` as it stands right after the `adult == False` filter (1,256,832 rows) -- does **not** carry forward the later in this notebook; both are superseded by the systematic exploration below. Works on a separate `tmdb_base` snapshot so `tmdb_dropped` itself is untouched and the rest of the notebook still runs unaffected.

### Drop `adult`/`popularity` as columns

`popularity` is excluded as leakage-adjacent (decision made earlier); `adult` already served its purpose as a row filter above and adds nothing further as a column now that every remaining row is `adult == False`.

In [ ]:
tmdb_base = tmdb_dropped.copy()
tmdb_base = tmdb_base.drop(columns=["adult", "popularity"])
print(f"tmdb_base: {tmdb_base.shape}")

### Build 7 candidate DataFrames

In [ ]:
tmdb_no_nulls = tmdb_base.dropna(how="any")

tmdb_budget_revenue_gt0 = tmdb_base.dropna(subset=["budget", "revenue"])
tmdb_budget_revenue_gt0 = tmdb_budget_revenue_gt0[tmdb_budget_revenue_gt0["vote_count"] > 0]

tmdb_budget_revenue_gt5 = tmdb_base.dropna(subset=["budget", "revenue"])
tmdb_budget_revenue_gt5 = tmdb_budget_revenue_gt5[tmdb_budget_revenue_gt5["vote_count"] > 5]

tmdb_budget_revenue_gt10 = tmdb_base.dropna(subset=["budget", "revenue"])
tmdb_budget_revenue_gt10 = tmdb_budget_revenue_gt10[tmdb_budget_revenue_gt10["vote_count"] > 10]

tmdb_no_budget_revenue_gt0 = tmdb_base.drop(columns=["budget", "revenue"])
tmdb_no_budget_revenue_gt0 = tmdb_no_budget_revenue_gt0[tmdb_no_budget_revenue_gt0["vote_count"] > 0]

tmdb_no_budget_revenue_gt5 = tmdb_base.drop(columns=["budget", "revenue"])
tmdb_no_budget_revenue_gt5 = tmdb_no_budget_revenue_gt5[tmdb_no_budget_revenue_gt5["vote_count"] > 5]

tmdb_no_budget_revenue_gt10 = tmdb_base.drop(columns=["budget", "revenue"])
tmdb_no_budget_revenue_gt10 = tmdb_no_budget_revenue_gt10[tmdb_no_budget_revenue_gt10["vote_count"] > 10]

### Shapes of all 7 candidates

In [ ]:
tmdb_candidates = {
    "tmdb_no_nulls": tmdb_no_nulls,
    "tmdb_budget_revenue_gt0": tmdb_budget_revenue_gt0,
    "tmdb_budget_revenue_gt5": tmdb_budget_revenue_gt5,
    "tmdb_budget_revenue_gt10": tmdb_budget_revenue_gt10,
    "tmdb_no_budget_revenue_gt0": tmdb_no_budget_revenue_gt0,
    "tmdb_no_budget_revenue_gt5": tmdb_no_budget_revenue_gt5,
    "tmdb_no_budget_revenue_gt10": tmdb_no_budget_revenue_gt10,
}
for name, df in tmdb_candidates.items():
    print(f"{name}: {df.shape}")

### Missingness of all 7 candidates

In [ ]:
for name, df in tmdb_candidates.items():
    missing = pd.DataFrame({
        "n_missing": df.isnull().sum(),
        "pct_missing": df.isnull().mean() * 100,
    }).sort_values("pct_missing", ascending=False)
    print(f"--- {name} ---")
    display(missing)

## End of candidate DataFrames section -- resuming main `tmdb_dropped` pipeline 

(does not use `tmdb_base` or the 7 candidates above)

### Top 20 keywords & co-occurrence

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

keywords_freq = tmdb_dropped["keywords_list"].explode().value_counts()
top20_keywords = keywords_freq.head(20)
top20_keywords

In [ ]:
top_items = top20_keywords.index.tolist()
restricted = tmdb_dropped["keywords_list"].apply(lambda items: [i for i in items if i in top_items])
mlb = MultiLabelBinarizer(classes=top_items)
onehot = mlb.fit_transform(restricted)
keywords_cooccurrence = pd.DataFrame(onehot.T @ onehot, index=top_items, columns=top_items)
keywords_cooccurrence

### `imdb_id` exact-duplicate check (all features except `id`)

Stricter than the duplicate-`imdb_id` check in Stage 1: here, a group only counts as a true duplicate if every other feature matches exactly (not just a handful of key columns). `id` is excluded from the comparison since it's TMDB's own row identifier and legitimately differs even for a genuine duplicate; the `*_list` columns are converted to tuples first since lists aren't hashable for a uniqueness comparison.

In [ ]:
list_cols_present = [c for c in [
    "genres_list", "keywords_list", "production_companies_list",
    "production_countries_list", "spoken_languages_list",
] if c in tmdb_dropped.columns]
compare_cols = [c for c in tmdb_dropped.columns if c not in ("id", "imdb_id")]

tmdb_imdb_check = tmdb_dropped[tmdb_dropped["imdb_id"].notna()].copy()
for c in list_cols_present:
    tmdb_imdb_check[c] = tmdb_imdb_check[c].apply(tuple)

dupe_imdb_mask = tmdb_imdb_check["imdb_id"].duplicated(keep=False)
tmdb_dupe_imdb_rows = tmdb_imdb_check[dupe_imdb_mask]

def fully_identical(group):
    return group[compare_cols].nunique(dropna=False).eq(1).all()

n_identical_groups = tmdb_dupe_imdb_rows.groupby("imdb_id").filter(fully_identical)["imdb_id"].nunique()
n_total_dupe_groups = tmdb_dupe_imdb_rows["imdb_id"].nunique()
print(f"{n_identical_groups} of {n_total_dupe_groups} duplicate-imdb_id groups "
      "have identical values across all other features (excluding id)")

### Missing values (after adult filter)

In [ ]:
tmdb_missing_after_adult_filter = pd.DataFrame({
    "n_missing": tmdb_dropped.isnull().sum(),
    "pct_missing": tmdb_dropped.isnull().mean() * 100,
}).sort_values("pct_missing", ascending=False)
tmdb_missing_after_adult_filter

### Drop records where `vote_count == 0` but `popularity != 0`

In [ ]:
tmdb_shape_before_popularity_fix = tmdb_dropped.shape
bad_popularity_mask = (tmdb_dropped["vote_count"] == 0) & (tmdb_dropped["popularity"] != 0)
tmdb_dropped = tmdb_dropped[~bad_popularity_mask]
print(f"Dropped {int(bad_popularity_mask.sum())} rows with vote_count==0 but popularity!=0")
print(f"Shape: {tmdb_shape_before_popularity_fix} -> {tmdb_dropped.shape}")

### Feature names & dtypes (final)

In [ ]:
tmdb_dropped.dtypes

## Airbnb feature engineering

### Feature names & dtypes

In [ ]:
airbnb_dropped.dtypes

### Remove duplicate ids

**DECISION POINT**: uses a plain `keep="first"` tiebreak, unlike TMDB's completeness-based dedup in Stage 1 -- no tiebreak rule was specified here. Revisit if the two datasets should be handled consistently.

In [ ]:
airbnb_shape_before_dedup = airbnb_dropped.shape
airbnb_dropped = airbnb_dropped.drop_duplicates(subset="id", keep="first")
print(f"Dropped {airbnb_shape_before_dedup[0] - airbnb_dropped.shape[0]} duplicate-id rows")
print(f"Shape: {airbnb_shape_before_dedup} -> {airbnb_dropped.shape}")

### Drop `amenities` superseded by `amenities_list`

In [ ]:
if "amenities_list" in airbnb_dropped.columns and "amenities" in airbnb_dropped.columns:
    airbnb_dropped = airbnb_dropped.drop(columns=["amenities"])
    print("Dropped 'amenities' (superseded by 'amenities_list')")
print(f"Airbnb shape now: {airbnb_dropped.shape}")

### Correlation matrix (Pearson & Spearman)

In [ ]:
airbnb_numeric = airbnb_dropped.select_dtypes(include="number")
airbnb_corr_pearson = airbnb_numeric.corr(method="pearson")
airbnb_corr_spearman = airbnb_numeric.corr(method="spearman")
print(f"{airbnb_numeric.shape[1]} numeric columns")
airbnb_corr_pearson

In [ ]:
n = airbnb_numeric.shape[1]
fig, ax = plt.subplots(figsize=(max(8, n * 0.5), max(7, n * 0.45)))
sns.heatmap(airbnb_corr_pearson, annot=False, cmap="coolwarm", center=0, ax=ax)
ax.set_title("Airbnb: numeric feature correlation matrix (Pearson)")
savefig(fig, "04_airbnb_correlation_heatmap_pearson.png")
plt.show()

In [ ]:
airbnb_corr_spearman

In [ ]:
fig, ax = plt.subplots(figsize=(max(8, n * 0.5), max(7, n * 0.45)))
sns.heatmap(airbnb_corr_spearman, annot=False, cmap="coolwarm", center=0, ax=ax)
ax.set_title("Airbnb: numeric feature correlation matrix (Spearman)")
savefig(fig, "04_airbnb_correlation_heatmap_spearman.png")
plt.show()

### Convert boolean/categorical-looking fields to `category`

NaN is filled with an explicit `"Missing"` level rather than left as an invisible gap, consistent with this project's general practice of treating missingness as its own informative category rather than silently dropping it.

In [ ]:
AIRBNB_CATEGORY_COLS = [
    "host_is_superhost", "host_has_profile_pic", "host_identity_verified",
    "source", "has_availability", "property_type", "room_type",
]
for col in AIRBNB_CATEGORY_COLS:
    airbnb_dropped[col] = airbnb_dropped[col].fillna("Missing").astype("category")
print(airbnb_dropped[AIRBNB_CATEGORY_COLS].dtypes)
print(airbnb_dropped[AIRBNB_CATEGORY_COLS].nunique())

### Convert `first_review`/`last_review` to datetime

In [ ]:
airbnb_dropped["first_review"] = pd.to_datetime(airbnb_dropped["first_review"], errors="coerce")
airbnb_dropped["last_review"] = pd.to_datetime(airbnb_dropped["last_review"], errors="coerce")
airbnb_dropped[["first_review", "last_review"]].dtypes

### Missing values

In [ ]:
airbnb_missing = pd.DataFrame({
    "n_missing": airbnb_dropped.isnull().sum(),
    "pct_missing": airbnb_dropped.isnull().mean() * 100,
}).sort_values("pct_missing", ascending=False)
airbnb_missing

In [ ]:
airbnb_dropped.head(10)

### Feature names & dtypes (final)

In [ ]:
airbnb_dropped.dtypes

In [ ]:
airbnb_dropped.shape